# Parquet Data Ingestion Pipeline using PySpark

## Objective
This notebook performs:

- Loading parquet data
- Schema inspection
- Data validation
- Year-wise and month-wise partitioning
- Ingestion into a second layer
- Ingestion reporting

The final output becomes ready for future data cleaning and transformation.

In [ ]:
# Install PySpark
!pip install pyspark

## Import Required Libraries

Import all libraries required for Spark processing and data transformation.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

SparkSession is the entry point for working with PySpark.

In [ ]:
# Start Spark session
spark = SparkSession.builder \
    .appName("Parquet_Ingestion_Pipeline") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


## Define Dataset Path

Specify the location of the parquet dataset.

In [ ]:
# Path of parquet dataset
file_path = "/content/team_5 (5).parquet"

## Load Parquet Dataset

Read the complete parquet dataset into a Spark DataFrame.

In [ ]:
# Load parquet file
df = spark.read.parquet(file_path)

## Display Sample Data

View sample records to verify successful loading.

In [ ]:
# Display sample rows
df.show(10, truncate=False)

+----------+-----+-----+--------------------------------------+-------------------------------+-------------------+----+----------+------+------+-----+---------+--------+-------+-------+---------+-----+--------------------------------------+----+-----+---+----+
|station_id|state|city |station_name                          |timestamp                      |datetime           |at_c|rh_percent|ws_m_s|wd_deg|rf_mm|tot_rf_mm|sr_w_mt2|bp_mmhg|vws_m_s|pollutant|value|station                               |year|month|day|hour|
+----------+-----+-----+--------------------------------------+-------------------------------+-------------------+----+----------+------+------+-----+---------+--------+-------+-------+---------+-----+--------------------------------------+----+-----+---+----+
|site_1424 |Delhi|Delhi|Jawaharlal Nehru Stadium, Delhi - DPCC|2024-01-01T00:00:00.000000+0000|2024-01-01 00:00:00|10.3|80.0      |1.1   |79.0  |0.0  |0.0      |5.0     |987.2  |NULL   |benzene  |1.7  |Jawaharlal N

## Print Schema

Display column names and data types.

In [ ]:
# Print dataset schema
df.printSchema()

root
 |-- station_id: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- at_c: double (nullable = true)
 |-- rh_percent: double (nullable = true)
 |-- ws_m_s: double (nullable = true)
 |-- wd_deg: double (nullable = true)
 |-- rf_mm: double (nullable = true)
 |-- tot_rf_mm: double (nullable = true)
 |-- sr_w_mt2: double (nullable = true)
 |-- bp_mmhg: double (nullable = true)
 |-- vws_m_s: double (nullable = true)
 |-- pollutant: string (nullable = true)
 |-- value: double (nullable = true)
 |-- station: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)



## Dataset Statistics

Count total rows and columns.

In [ ]:
# Count rows and columns
row_count = df.count()
column_count = len(df.columns)

print(f"Total Rows    : {row_count}")
print(f"Total Columns : {column_count}")

Total Rows    : 8844800
Total Columns : 22


## Identify Date Column

Check available columns and identify the date column required for partitioning.

In [ ]:
# Print all columns
print(df.columns)

['station_id', 'state', 'city', 'station_name', 'timestamp', 'datetime', 'at_c', 'rh_percent', 'ws_m_s', 'wd_deg', 'rf_mm', 'tot_rf_mm', 'sr_w_mt2', 'bp_mmhg', 'vws_m_s', 'pollutant', 'value', 'station', 'year', 'month', 'day', 'hour']


## Convert Date Column

Convert the date column into proper date format.

Replace `inspection_date` with your actual date column if different.

In [ ]:
# Convert datetime column to date
df = df.withColumn(
    "datetime",
    col("datetime").cast("date")
)

# Create year column
df = df.withColumn(
    "year",
    year(col("datetime"))
)

# Create month column
df = df.withColumn(
    "month",
    month(col("datetime"))
)

In [ ]:
df.select(
    "datetime",
    "year",
    "month"
).show(20, truncate=False)

+----------+----+-----+
|datetime  |year|month|
+----------+----+-----+
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
|2024-01-01|2024|1    |
+----------+----+-----+
only showing top 20 rows


## Validate Missing Values

Check null values across all columns before ingestion.

In [ ]:
# Count null values in each column
null_check = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_check.show(truncate=False)

+----------+-----+----+------------+---------+--------+-------+----------+-------+-------+-------+---------+--------+-------+-------+---------+-----+-------+----+-----+---+----+
|station_id|state|city|station_name|timestamp|datetime|at_c   |rh_percent|ws_m_s |wd_deg |rf_mm  |tot_rf_mm|sr_w_mt2|bp_mmhg|vws_m_s|pollutant|value|station|year|month|day|hour|
+----------+-----+----+------------+---------+--------+-------+----------+-------+-------+-------+---------+--------+-------+-------+---------+-----+-------+----+-----+---+----+
|0         |0    |0   |0           |0        |0       |2532923|1519086   |1531562|1836378|4006105|801929   |1610725 |3022601|6546511|0        |0    |0      |0   |0    |0  |0   |
+----------+-----+----+------------+---------+--------+-------+----------+-------+-------+-------+---------+--------+-------+-------+---------+-----+-------+----+-----+---+----+



## Remove Duplicate Records

Remove duplicate rows to improve data quality before ingestion.

In [ ]:
# Count rows before removing duplicates
before_count = df.count()

# Remove duplicate rows
df = df.dropDuplicates()

# Count rows after removing duplicates
after_count = df.count()

print(f"Rows Before Deduplication : {before_count}")
print(f"Rows After Deduplication  : {after_count}")
print(f"Duplicates Removed        : {before_count - after_count}")

Rows Before Deduplication : 8844800
Rows After Deduplication  : 8844800
Duplicates Removed        : 0


## Define Output Path

Specify the storage location for the ingestion layer.

In [ ]:
# Output path for partitioned data
output_path = "/content/ingestion_layer/partitioned_data"

## Write Partitioned Data

Store the dataset partitioned by year and month.

In [ ]:
# Write partitioned parquet dataset
df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(output_path)

print("Partitioned Data Written Successfully")

Partitioned Data Written Successfully


## Read Ingested Data

Load the partitioned dataset to verify successful ingestion.

In [ ]:
# Read partitioned parquet data
partitioned_df = spark.read.parquet(output_path)

partitioned_df.show(10, truncate=False)

+----------+-----+-----+-----------------------------+-------------------------------+----------+----+----------+------+------+-----+---------+--------+-------+-------+---------+------+-----------------------------+---+----+----+-----+
|station_id|state|city |station_name                 |timestamp                      |datetime  |at_c|rh_percent|ws_m_s|wd_deg|rf_mm|tot_rf_mm|sr_w_mt2|bp_mmhg|vws_m_s|pollutant|value |station                      |day|hour|year|month|
+----------+-----+-----+-----------------------------+-------------------------------+----------+----+----------+------+------+-----+---------+--------+-------+-------+---------+------+-----------------------------+---+----+----+-----+
|site_1427 |Delhi|Delhi|Najafgarh, Delhi - DPCC      |2024-03-01T00:00:00.000000+0000|2024-03-01|14.0|78.0      |0.4   |210.0 |0.0  |0.0      |3.0     |994.0  |NULL   |pm10     |178.0 |Najafgarh, Delhi - DPCC      |1  |0   |2024|3    |
|site_105  |Delhi|Delhi|North Campus, DU, Delhi - IMD|20

## Verify Partitions

Display available year-wise and month-wise partitions.

In [ ]:
# Show distinct partitions
partitioned_df.select(
    "year",
    "month"
).distinct().orderBy(
    "year",
    "month"
).show(100)

+----+-----+
|year|month|
+----+-----+
|2024|    1|
|2024|    2|
|2024|    3|
|2024|    4|
|2024|    5|
|2024|    6|
|2024|    7|
|2024|    8|
|2024|    9|
|2024|   10|
|2024|   11|
|2024|   12|
|2025|    1|
|2025|    2|
|2025|    3|
|2025|    4|
|2025|    5|
|2025|    6|
|2025|    7|
|2025|    8|
|2025|    9|
|2025|   10|
|2025|   11|
|2025|   12|
+----+-----+



## Validate Final Row Count

Ensure no records were lost during ingestion.

In [ ]:
# Final row count validation
final_count = partitioned_df.count()

print(f"Original Row Count : {row_count}")
print(f"Final Row Count    : {final_count}")

Original Row Count : 8844800
Final Row Count    : 8844800


## Ingestion Summary

Display final ingestion statistics.

In [ ]:
# Display ingestion report
print("=" * 50)
print("INGESTION PIPELINE SUMMARY")
print("=" * 50)

print(f"Rows Processed      : {final_count}")
print(f"Columns             : {column_count}")
print(f"Partition Columns   : year, month")
print(f"Output Path         : {output_path}")
print(f"Duplicates Removed  : {before_count - after_count}")

print("=" * 50)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 50)

INGESTION PIPELINE SUMMARY
Rows Processed      : 8844800
Columns             : 22
Partition Columns   : year, month
Output Path         : /content/ingestion_layer/partitioned_data
Duplicates Removed  : 0
PIPELINE COMPLETED SUCCESSFULLY


## Stop Spark Session

Close the Spark session after successful execution.

In [ ]:
# Stop Spark session
spark.stop()

In [ ]:
# List partition folders
import os

for root, dirs, files in os.walk(output_path):
    print(root)

/content/ingestion_layer/partitioned_data
/content/ingestion_layer/partitioned_data/year=2024
/content/ingestion_layer/partitioned_data/year=2024/month=9
/content/ingestion_layer/partitioned_data/year=2024/month=3
/content/ingestion_layer/partitioned_data/year=2024/month=4
/content/ingestion_layer/partitioned_data/year=2024/month=6
/content/ingestion_layer/partitioned_data/year=2024/month=10
/content/ingestion_layer/partitioned_data/year=2024/month=1
/content/ingestion_layer/partitioned_data/year=2024/month=8
/content/ingestion_layer/partitioned_data/year=2024/month=5
/content/ingestion_layer/partitioned_data/year=2024/month=12
/content/ingestion_layer/partitioned_data/year=2024/month=7
/content/ingestion_layer/partitioned_data/year=2024/month=2
/content/ingestion_layer/partitioned_data/year=2024/month=11
/content/ingestion_layer/partitioned_data/year=2025
/content/ingestion_layer/partitioned_data/year=2025/month=9
/content/ingestion_layer/partitioned_data/year=2025/month=3
/content/in

# Conclusion

A complete parquet ingestion pipeline was successfully implemented using PySpark. The dataset was partitioned year-wise and month-wise and stored into an ingestion layer. The final output is optimized for future cleaning, transformation, and analytical processing.

## Why Partitioning is Important

Partitioning improves query performance by reducing the amount of data scanned. Instead of reading the entire dataset, Spark reads only the required partitions, making big data processing faster and more efficient.

In [27]:
# Zip the ingestion layer folder
!zip -r ingestion_layer.zip ingestion_layer

  adding: ingestion_layer/ (stored 0%)
  adding: ingestion_layer/partitioned_data/ (stored 0%)
  adding: ingestion_layer/partitioned_data/year=2024/ (stored 0%)
  adding: ingestion_layer/partitioned_data/year=2024/month=9/ (stored 0%)
  adding: ingestion_layer/partitioned_data/year=2024/month=9/part-00007-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet (deflated 19%)
  adding: ingestion_layer/partitioned_data/year=2024/month=9/.part-00004-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet.crc (stored 0%)
  adding: ingestion_layer/partitioned_data/year=2024/month=9/.part-00001-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet.crc (stored 0%)
  adding: ingestion_layer/partitioned_data/year=2024/month=9/part-00005-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet (deflated 17%)
  adding: ingestion_layer/partitioned_data/year=2024/month=9/.part-00003-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet.crc (stored 0%)
  adding: ingestion_layer/partitioned_d

## Generate Ingestion Logs

Create a log file containing ingestion statistics.

In [28]:
# Create ingestion log file

log_text = f"""
INGESTION PIPELINE LOG
======================

Rows Processed      : {final_count}
Columns             : {column_count}
Partition Columns   : year, month
Output Path         : {output_path}
Duplicates Removed  : {before_count - after_count}

Pipeline Status     : SUCCESS
"""

with open("ingestion_logs.txt", "w") as file:
    file.write(log_text)

print("Log File Created Successfully")

Log File Created Successfully
